<a href="https://colab.research.google.com/github/taskeen-01/Blinkit-Sales-Analysis_Jupyter/blob/main/InsightGen_%E2%80%93_AI_Powered_Order_Analytics_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**InsightGen: AI-Powered Order Analytics Chatbot**

Built a Retrieval-Augmented Generation (RAG) system using FAISS, Sentence Transformers, and Gemini to answer order-related queries from PDF data.

It is an Order Analytics chatbot using a Retrieval-Augmented Generation (RAG) architecture.

I extracted order data from PDF files using Tabula, converted the data into textual documents, and generated embeddings using Sentence Transformers.

These embeddings were stored in FAISS for efficient similarity search.

When a user asks a question, the system retrieves the most relevant order data and sends it along with the query to the Gemini LLM to generate a contextual response.

**Define PDF Path**

This step defines the file path of the PDF document that contains the order data. The PDF will later be used for extracting tables.

In [2]:
# Path of the PDF file that contains order data
path = '/content/order.pdf'

**Install Required Libraries**

These libraries are required for the project:

tabula-py → Extract tables from PDF

faiss-cpu → Fast similarity search

sentence-transformers → Create embeddings

google-generativeai → Use Gemini LLM

In [19]:
#I do this separtely because it is causing problem in installing with other libraries
!pip install tabula-py

In [6]:
# Install required libraries

!pip install tabula-py
!pip install faiss-cpu
!pip install sentence-transformers
!pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.5 MB/s eta 0:00:00


**Import Libraries**

This step imports all the necessary Python libraries used for:

1. Data processing
2. Embedding generation
3. Vector search
4. LLM interaction

In [8]:
# Import necessary libraries

import tabula                      # Extract tables from PDF
import pandas as pd                # Data manipulation
import numpy as np                 # Numerical operations
import faiss                       # Vector similarity search
from sentence_transformers import SentenceTransformer  # Embedding model
import google.generativeai as genai  # Gemini API
from google.colab import userdata  # Access Colab secrets

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


**Load Gemini API Key**

This step loads the Gemini API key stored in Colab secrets and configures the Gemini model for use.

In [9]:
# Load Gemini API key from Colab secrets

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Check if API key exists
if GEMINI_API_KEY is None:
    raise ValueError('GEMINI_API_KEY not found in Colab Secrets')

# Configure Gemini API
genai.configure(api_key=GEMINI_API_KEY)

**Extract Tables from PDF**

This step extracts tables from the PDF file using Tabula and converts them into a Pandas DataFrame.

In [11]:
# Extract tables from PDF

pdf_path = '/content/order.pdf'

# Read tables from all pages
tables = tabula.read_pdf(pdf_path, pages='all')

# Combine all tables into one DataFrame
df = pd.concat(tables, ignore_index=True)

# Display extracted data
df

,order_id,order_status,customer,order_date,order_quantity,sales
0,3,Order\rFinished,Muhammed Mac\rIntyre,13/10/2010,6,523080
1,293,Order\rFinished,Barry French,1/10/2012,49,20246040
2,483,Order\rFinished,Clay Rozendal,10/7/2011,30,9931519
3,515,Order\rFinished,Carlos Soltero,28/8/2010,19,788540
4,613,Order\rFinished,Carl Jackson,17/6/2011,12,187080
5,643,Order\rFinished,Monica Federle,24/3/2011,21,5563640
6,678,Order\rReturned,Dorothy Badders,26/2/2010,44,456820
7,807,Order\rFinished,Neola Schneider,23/11/2010,45,393700
8,868,Order\rFinished,Carlos Daly,8/6/2012,32,1433680
9,933,Order\rFinished,Claudia Miner,4/8/2012,15,161220


**Convert Data into Text Documents**

Each row of the dataset is converted into natural language text.
These text documents will later be converted into embeddings.

This helps the LLM understand the data better.

In [12]:
# Convert each row of the dataframe into text format

documents = []

for _, row in df.iterrows():

    doc = (
        f"Order ID: {row['order_id']} has status {row['order_status']}\n"
        f"Customer is {row['customer']}."
        f"Order Date: {row['order_date']}."
        f"Quantity Ordered is {row['order_quantity']}."
        f"Total Sales amount is {row['sales']}."
    )

    # Add document to list
    documents.append(doc)

# Print generated documents
print(documents)

['Order ID: 3 has status Order\rFinished\nCustomer is Muhammed Mac\rIntyre.Order Date: 13/10/2010.Quantity Ordered is 6.Total Sales amount is 523080.', 'Order ID: 293 has status Order\rFinished\nCustomer is Barry French.Order Date: 1/10/2012.Quantity Ordered is 49.Total Sales amount is 20246040.', 'Order ID: 483 has status Order\rFinished\nCustomer is Clay Rozendal.Order Date: 10/7/2011.Quantity Ordered is 30.Total Sales amount is 9931519.', 'Order ID: 515 has status Order\rFinished\nCustomer is Carlos Soltero.Order Date: 28/8/2010.Quantity Ordered is 19.Total Sales amount is 788540.', 'Order ID: 613 has status Order\rFinished\nCustomer is Carl Jackson.Order Date: 17/6/2011.Quantity Ordered is 12.Total Sales amount is 187080.', 'Order ID: 643 has status Order\rFinished\nCustomer is Monica Federle.Order Date: 24/3/2011.Quantity Ordered is 21.Total Sales amount is 5563640.', 'Order ID: 678 has status Order\rReturned\nCustomer is Dorothy Badders.Order Date: 26/2/2010.Quantity Ordered is 4

**Generate Embeddings**

Embeddings convert text into numerical vectors so machines can understand semantic meaning.

The SentenceTransformer model generates embeddings for each document.

In [13]:
# Load embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings for documents
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Print embeddings
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-0.03857804 -0.03845885  0.05559022 ... -0.08661829 -0.04345511
   0.05286995]
 [-0.0308831  -0.02727907 -0.00534819 ... -0.05232951 -0.05835786
  -0.00960175]
 [-0.06071752 -0.03426303  0.00897049 ... -0.03492578 -0.07255565
  -0.01401187]
 ...
 [ 0.01966314 -0.00308781  0.07407957 ... -0.03670935 -0.01944768
  -0.02848411]
 [-0.09005257  0.01092917  0.04139737 ... -0.04200085 -0.00766683
   0.00255755]
 [-0.06571691 -0.01627015  0.00909008 ... -0.01118064 -0.08724661
   0.01612516]]


**Store Embeddings in FAISS**

FAISS is used to store embeddings in a vector database.
This allows fast similarity search when a user asks a question.

In [14]:
# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embeddings)

# Save FAISS index file
faiss.write_index(index, 'faiss_index.faiss')

**Create Retrieval Function**

This function retrieves the most relevant documents based on the user query using similarity search.

In [15]:
# Function to retrieve relevant context from FAISS

def retrieve_context(query, k=3):

    # Convert query into embedding
    query_embedding = embedding_model.encode([query])

    # Search similar vectors
    distances, indices = index.search(query_embedding, k)

    # Return matching documents
    return '\n'.join([documents[i] for i in indices[0]])

**Configure Gemini Model**

This step configures the Gemini model parameters:

temperature → creativity of response

max_output_tokens → maximum response length

In [16]:
# Gemini generation configuration

generation_config = {
    "temperature": 0.4,       # Controls creativity
    "max_output_tokens": 512  # Maximum response length
}

print(generation_config)

# Initialize Gemini model
gemini_model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    generation_config=generation_config
)

{'temperature': 0.4, 'max_output_tokens': 512}


**Chatbot Function**

This function:
1. Retrieves relevant order data
2. Sends context + question to Gemini
3. Returns the generated response

In [17]:
# Conversational chatbot function

chat_history = []

def chat_with_bot(user_input):

    global chat_history

    # Retrieve relevant context
    context = retrieve_context(user_input)

    # Create prompt
    prompt = f"""
    You are a helpful conversational data analyst assistant.

    Context:
    {context}

    User Question:
    {user_input}

    Rules:
    - Be conversational
    - Answer only using the context
    - If you don't know the answer, say you don't have enough information
    """

    # Generate response from Gemini
    response = gemini_model.generate_content(prompt)

    return response.text

**Run Chatbot Loop**

This creates an interactive chatbot where users can ask questions about order data.

In [18]:
print("Order Analytics Chat Bot Ready !!!")
print("Type 'exit' to stop \n ")

while True:

    # Take user input
    user_input = input("User: ")

    # Exit condition
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("Good Bye")
        break

    # Generate response
    response = chat_with_bot(user_input)

    # Print bot response
    print(f"Bot: {response}")
    print("-"*60)

Order Analytics Chat Bot Ready !!!
Type 'exit' to stop 
 
User: What is the status of order 1001?
Bot: I'm sorry, but I don't have any information about the status of order 1001 in the context provided. I only have details for orders 998, 1154, and 995.
------------------------------------------------------------
User: What is the total sales?
Bot: Based on the information provided, the total sales amount is 2,528,828.
------------------------------------------------------------
User: What is the average sales?
Bot: Based on the information provided, the total sales amounts are 1,669,808, 740,960, and 118,060.

To find the average sales, we sum these amounts and divide by the number of orders (3):
(1,669,808 + 740,960 + 118,060) / 3 = 2,528,828 / 3 = 842,942.67

So, the average sales amount is 842,942.67.
------------------------------------------------------------
User: exit
Good Bye


**Important: Accessing the Gemini API Key**

The Gemini API key is required to connect Python with the Gemini LLM.
Instead of writing the API key directly in the code (which is unsafe), it is stored securely in Google Colab Secrets.

This method keeps the API key private and secure.

**Process**
1. Go to Google Colab
2. Click 🔑 Secrets (left sidebar)
3. Add a new secret
4. Name it GEMINI_API_KEY
5. Paste your Gemini API key
6. Save it
7. Then the code retrieves the key securely using userdata.get()